# SQL Queries — `data_engineering` Catalog

Connects to the Databricks SQL warehouse using the `school` profile from `~/.databrickscfg` and runs one query per table across all three medallion layers.

| Layer | Tables |
|---|---|
| Bronze | `brfss_2024`, `medicaid_expansion_status_raw`, `svi_2022_us_county` |
| Silver | `brfss_2024_transformed`, `medicaid_expansion_clean_transformed`, `svi_2022_state_transformed` |
| Gold | `dim_respondent`, `dim_behavior`, `dim_chronic_condition`, `dim_healthcare_access`, `dim_preventive_care`, `dim_location`, `dim_medicaid`, `dim_svi`, `dim_time`, `fact_health_response` |

In [1]:
import configparser
import os

import pandas as pd
from databricks import sql

# Read credentials from ~/.databrickscfg [school] profile
_cfg = configparser.ConfigParser()
_cfg.read(os.path.expanduser("~/.databrickscfg"))
_profile = _cfg["school"]

HOST      = _profile["host"].replace("https://", "").rstrip("/")
TOKEN     = _profile["token"]
HTTP_PATH = "/sql/1.0/warehouses/8a0ddda80f05d456"

conn = sql.connect(
    server_hostname=HOST,
    http_path=HTTP_PATH,
    access_token=TOKEN,
)

def run_query(sql_text: str) -> pd.DataFrame:
    with conn.cursor() as cur:
        cur.execute(sql_text)
        rows = cur.fetchall()
        cols = [c[0] for c in cur.description]
    return pd.DataFrame(rows, columns=cols)

print(f"Connected to {HOST}")

[WARN] pyarrow is not installed by default since databricks-sql-connector 4.0.0,any arrow specific api (e.g. fetchmany_arrow) and cloud fetch will be disabled.If you need these features, please run pip install pyarrow or pip install databricks-sql-connector[pyarrow] to install


Connected to dbc-670721c9-ce87.cloud.databricks.com


---
## Bronze Layer

In [2]:
# bronze.brfss_2024 — raw BRFSS 2024 survey responses loaded from CDC source without transformation
# Samples 20 rows to inspect raw CDC-coded field names before cleaning: state FIPS (_STATE),
# collection period (IYEAR, IMONTH), demographics (SEXVAR, _AGEG5YR, _RACE), self-reported
# general and mental/physical health (GENHLTH, PHYSHLTH, MENTHLTH), binary chronic condition
# flags (CVDINFR4=heart attack, CVDSTRK3=stroke, DIABETE4=diabetes, ADDEPEV3=depression),
# BMI category (_BMI5CAT), and the raked complex survey weight (_LLCPWT) that must be applied
# in all downstream prevalence calculations to produce population-representative estimates.
run_query("""
SELECT
    _STATE,
    IYEAR,
    IMONTH,
    SEXVAR,
    _AGEG5YR,
    _RACE,
    GENHLTH,
    PHYSHLTH,
    MENTHLTH,
    _BMI5CAT,
    CVDINFR4,
    CVDSTRK3,
    DIABETE4,
    ADDEPEV3,
    _LLCPWT
FROM data_engineering.bronze.brfss_2024
LIMIT 20
""")

,_STATE,IYEAR,IMONTH,SEXVAR,_AGEG5YR,_RACE,GENHLTH,PHYSHLTH,MENTHLTH,_BMI5CAT,CVDINFR4,CVDSTRK3,DIABETE4,ADDEPEV3,_LLCPWT
0,1.0,2024,02,2.0,12.0,1.0,3.0,2.0,88.0,2.0,2.0,2.0,3.0,2.0,261.525511
1,1.0,2024,02,1.0,13.0,1.0,1.0,88.0,88.0,3.0,2.0,2.0,3.0,2.0,307.169688
2,1.0,2024,02,1.0,8.0,1.0,2.0,30.0,88.0,2.0,2.0,2.0,3.0,2.0,2939.862806
3,1.0,2024,02,1.0,13.0,1.0,1.0,88.0,88.0,3.0,2.0,2.0,3.0,2.0,153.584844
4,1.0,2024,02,1.0,6.0,1.0,3.0,88.0,88.0,2.0,2.0,2.0,3.0,2.0,1229.623036
5,1.0,2024,02,1.0,7.0,1.0,3.0,7.0,88.0,4.0,2.0,2.0,1.0,2.0,451.219871
6,1.0,2024,02,2.0,11.0,1.0,4.0,88.0,88.0,4.0,1.0,2.0,3.0,2.0,388.467984
7,1.0,2024,02,2.0,10.0,1.0,5.0,30.0,88.0,4.0,2.0,1.0,1.0,2.0,194.233992
8,1.0,2024,02,2.0,11.0,1.0,3.0,5.0,88.0,4.0,2.0,2.0,3.0,2.0,455.584194
9,1.0,2024,02,1.0,13.0,1.0,4.0,14.0,88.0,2.0,2.0,2.0,1.0,2.0,211.232063


In [3]:
# bronze.medicaid_expansion_status_raw — raw Kaiser Family Foundation Medicaid expansion tracker
# Full table loaded directly from KFF's state-level ACA Medicaid expansion dataset showing
# each state's adoption decision and, where applicable, the year coverage took effect.
# This is the policy ground truth used to classify states in all downstream Medicaid analysis;
# non-expanding states serve as the counterfactual in health outcome comparisons.
run_query("""
SELECT *
FROM data_engineering.bronze.medicaid_expansion_status_raw
ORDER BY Location
""")

,Location,Status of Medicaid Expansion Decision,Expansion Implementation Date,Expansion Adopted Through Ballot Initiative,Trigger Law In Place,Footnotes
0,Alabama,Not Adopted,N/A,N/A,None,None
1,Alaska,Adopted,9/1/15,No,No,None
2,Arizona,Adopted,1/1/14,No,Yes,1
3,Arkansas,Adopted,1/1/14,No,Yes,1
4,California,Adopted,1/1/14,No,No,None
5,Colorado,Adopted,1/1/14,No,No,None
6,Connecticut,Adopted,1/1/14,No,No,None
7,Delaware,Adopted,1/1/14,No,No,None
8,District of Columbia,Adopted,1/1/14,No,No,None
9,Florida,Not Adopted,N/A,N/A,None,None


In [4]:
# bronze.svi_2022_us_county — CDC Social Vulnerability Index 2022 at county level, pre-aggregation
# Surfaces the 20 most vulnerable counties by overall SVI percentile rank (RPL_THEMES = 1.0 is most
# vulnerable) alongside the four CDC theme subscores: socioeconomic status (RPL_THEME1), household
# composition & disability (RPL_THEME2), minority & language access (RPL_THEME3), and
# housing & transportation (RPL_THEME4). Includes county population, uninsured rate, and poverty
# rate as key social determinants. This county-grain data is later aggregated to state level
# in the silver layer for joining with the state-level BRFSS and Medicaid datasets.
run_query("""
SELECT
    STATE,
    COUNTY,
    FIPS,
    E_TOTPOP,
    RPL_THEMES  AS svi_overall,
    RPL_THEME1  AS svi_socioeconomic,
    RPL_THEME2  AS svi_household_composition,
    RPL_THEME3  AS svi_minority_language,
    RPL_THEME4  AS svi_housing_transport,
    EP_UNINSUR  AS pct_uninsured,
    EP_POV150   AS pct_poverty
FROM data_engineering.bronze.svi_2022_us_county
WHERE RPL_THEMES >= 0
ORDER BY RPL_THEMES DESC
LIMIT 20
""")

,STATE,COUNTY,FIPS,E_TOTPOP,svi_overall,svi_socioeconomic,svi_household_composition,svi_minority_language,svi_housing_transport,pct_uninsured,pct_poverty
0,Louisiana,Madison Parish,22065,10028,1.0000,0.9663,0.9640,0.9459,1.0000,11.3,51.8
1,New Mexico,Luna County,35029,25393,0.9997,0.9577,1.0000,0.9704,0.9367,8.7,43.2
2,Texas,Dimmit County,48127,8672,0.9994,0.9812,0.9971,0.9940,0.9631,17.3,60.9
3,Louisiana,Evangeline Parish,22039,32335,0.9990,0.9841,0.9955,0.7261,0.9895,9.1,43.1
4,Arkansas,Chicot County,5017,10234,0.9987,0.9800,0.9866,0.9243,0.9889,9.8,43.1
5,North Carolina,Lenoir County,37107,55071,0.9984,0.9710,0.9975,0.8794,0.9774,10.8,36.9
6,Florida,DeSoto County,12027,34258,0.9981,0.9920,0.9656,0.8447,0.9876,19.6,40.1
7,Arizona,Yuma County,4027,204374,0.9978,0.9819,0.9736,0.9612,0.9800,13.5,31.7
8,Mississippi,Yazoo County,28163,27467,0.9975,0.9968,0.8396,0.9450,0.9952,13.7,42.0
9,Mississippi,Washington County,28151,44604,0.9971,0.9965,0.9784,0.9758,0.8836,13.4,43.2


---
## Silver Layer

In [5]:
# silver.brfss_2024_transformed — BRFSS data after cleaning, recoding, and label mapping
# Computes survey-weight-adjusted chronic condition and insurance coverage rates per state.
# Using survey weights (rather than raw counts) is essential for BRFSS: the CDC over-samples
# smaller states and certain demographic groups, so unweighted rates systematically misrepresent
# true population prevalence. Ordering by diabetes rate descending surfaces highest-burden states
# and allows visual comparison with Medicaid expansion and SVI status in later queries.
run_query("""
SELECT
    state_name,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CASE WHEN diabetes     = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS diabetes_pct,
    ROUND(SUM(CASE WHEN heart_attack = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS heart_attack_pct,
    ROUND(SUM(CASE WHEN stroke       = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS stroke_pct,
    ROUND(SUM(CASE WHEN depression   = TRUE THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS depression_pct,
    ROUND(SUM(CASE WHEN has_insurance = 'Yes' THEN survey_weight ELSE 0 END)
          / SUM(survey_weight) * 100, 2)                                       AS insured_pct
FROM data_engineering.silver.brfss_2024_transformed
WHERE state_name IS NOT NULL
GROUP BY state_name
ORDER BY diabetes_pct DESC
""")

,state_name,respondents,diabetes_pct,heart_attack_pct,stroke_pct,depression_pct,insured_pct
0,West Virginia,5816,18.34,6.98,4.92,30.07,0.0
1,Puerto Rico,4309,17.77,4.70,2.00,17.96,0.0
2,Virgin Islands,1348,17.72,3.31,4.78,14.20,0.0
3,Kentucky,7470,16.14,6.54,4.63,28.19,0.0
4,Louisiana,4547,15.38,5.08,5.16,26.39,0.0
5,Arkansas,5344,15.29,5.86,5.01,24.98,0.0
6,Mississippi,2986,15.18,4.55,4.99,19.53,0.0
7,Alabama,5092,15.04,5.51,4.71,24.69,0.0
8,Indiana,13050,14.20,5.25,4.21,24.36,0.0
9,South Carolina,9485,13.87,4.99,4.16,20.60,0.0


In [6]:
# silver.medicaid_expansion_clean_transformed — Medicaid expansion data after status standardization
# Groups states by expansion decision and shows the range of adoption years within each group.
# The earliest/latest year spread reveals how gradually expansion rolled out among adopting states
# (ACA-compliant expansion started in 2014; holdout states adopted as late as 2023), which matters
# for interpreting health outcome differences — states that expanded earlier had more years to
# realize coverage gains by the time the 2024 BRFSS data was collected.
run_query("""
SELECT
    medicaid_expansion_status,
    COUNT(*)            AS state_count,
    MIN(expansion_year) AS earliest_year,
    MAX(expansion_year) AS latest_year
FROM data_engineering.silver.medicaid_expansion_clean_transformed
GROUP BY medicaid_expansion_status
ORDER BY state_count DESC
""")

,medicaid_expansion_status,state_count,earliest_year,latest_year
0,Adopted,41,2014.0,2023.0
1,Not Adopted,10,NaN,NaN
2,Not Applicable,3,NaN,NaN


In [7]:
# silver.svi_2022_state_transformed — SVI aggregated from county to state level after transformation
# Ranks all 50 states and DC by overall SVI percentile (0 = least vulnerable, 1 = most vulnerable)
# and exposes all four CDC theme subscores alongside three key social determinants: uninsured rate,
# poverty rate, and unemployment rate. Ordering by svi_overall descending immediately surfaces the
# states with the greatest structural disadvantage, which tend to overlap with non-expanding Medicaid
# states — a relationship tested more rigorously in the gold layer cross-tabulation queries.
run_query("""
SELECT
    state_name,
    state_code,
    ROUND(svi_overall, 4)           AS svi_overall,
    ROUND(svi_socioeconomic, 4)     AS svi_socioeconomic,
    ROUND(svi_household, 4)         AS svi_household,
    ROUND(svi_minority, 4)          AS svi_minority,
    ROUND(svi_housing_transport, 4) AS svi_housing_transport,
    ROUND(pct_uninsured, 2)         AS pct_uninsured,
    ROUND(pct_poverty, 2)           AS pct_poverty,
    ROUND(pct_unemployed, 2)        AS pct_unemployed
FROM data_engineering.silver.svi_2022_state_transformed
ORDER BY svi_overall DESC
""")

,state_name,state_code,svi_overall,svi_socioeconomic,svi_household,svi_minority,svi_housing_transport,pct_uninsured,pct_poverty,pct_unemployed
0,New Mexico,NM,0.8152,0.7510,0.7838,0.9315,0.7013,9.52,29.15,6.34
1,Nevada,NV,0.8075,0.8161,0.6208,0.8699,0.7014,11.36,21.59,7.07
2,Louisiana,LA,0.7760,0.7300,0.6975,0.7777,0.7042,8.06,28.79,6.61
3,Arizona,AZ,0.7671,0.6713,0.6398,0.8415,0.7590,10.76,21.58,5.50
4,Texas,TX,0.7667,0.7502,0.6657,0.8883,0.6482,17.57,23.17,5.26
5,Mississippi,MS,0.7528,0.7688,0.6615,0.7777,0.6129,11.83,30.46,6.52
6,California,CA,0.7232,0.6705,0.3988,0.9267,0.7960,7.09,19.99,6.47
7,Florida,FL,0.7217,0.6970,0.5404,0.7991,0.6888,12.29,21.79,5.05
8,Oklahoma,OK,0.7199,0.6718,0.7233,0.7337,0.6142,13.93,25.03,4.91
9,Arkansas,AR,0.6796,0.6197,0.7111,0.6262,0.6230,8.83,27.30,5.22


---
## Gold Layer

In [8]:
# gold.dim_respondent — demographic dimension: one row per unique sex × age × race × income profile
# Cross-tabulates the four core BRFSS demographic attributes to show how survey respondents
# are distributed across demographic strata. This dimension is joined to fact_health_response
# to enable stratified analysis of disease burden by any combination of demographics.
# High respondent counts indicate the most commonly surveyed profiles and also the groups
# for which prevalence estimates will be most statistically reliable.
run_query("""
SELECT
    sex,
    age_group,
    race,
    income_group,
    COUNT(*) AS respondents
FROM data_engineering.gold.dim_respondent
WHERE sex IS NOT NULL
  AND age_group IS NOT NULL
GROUP BY sex, age_group, race, income_group
ORDER BY respondents DESC
LIMIT 30
""")

,sex,age_group,race,income_group,respondents
0,Female,80+,White NH,None,7081
1,Female,65-69,White NH,$50-100k,5978
2,Female,70-74,White NH,$50-100k,5668
3,Male,70-74,White NH,$50-100k,5324
4,Male,65-69,White NH,$50-100k,5227
5,Female,60-64,White NH,$50-100k,4929
6,Male,75-79,White NH,$50-100k,4593
7,Female,75-79,White NH,$50-100k,4517
8,Female,75-79,White NH,None,4381
9,Female,70-74,White NH,None,4352


In [9]:
# gold.dim_behavior — behavioral risk factor dimension: one row per unique lifestyle profile
# Cross-tabulates smoking status × BMI category to reveal how behavioral risk factors cluster:
# shows average BMI within each group and the share reporting exercise, binge drinking, and heavy
# drinking. This reveals co-occurrence patterns (e.g., current smokers with obese BMI also tend
# to have lower exercise rates), which are relevant for understanding the behavioral pathways
# through which social vulnerability translates into chronic disease in the fact table analysis.
run_query("""
SELECT
    smoke_status,
    bmi_category_clean,
    COUNT(*)                                                                   AS respondents,
    ROUND(AVG(CAST(bmi_raw AS DOUBLE)), 1)                                     AS avg_bmi,
    ROUND(SUM(CASE WHEN exercise       = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_exercise,
    ROUND(SUM(CASE WHEN binge_drinking = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_binge_drink,
    ROUND(SUM(CASE WHEN heavy_drinker  = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_heavy_drink
FROM data_engineering.gold.dim_behavior
WHERE smoke_status IS NOT NULL
  AND bmi_category_clean IS NOT NULL
GROUP BY smoke_status, bmi_category_clean
ORDER BY respondents DESC
""")

,smoke_status,bmi_category_clean,respondents,avg_bmi,pct_exercise,pct_binge_drink,pct_heavy_drink
0,Never,Overweight,84603,27.3,83.0,11.8,4.0
1,Never,Obese,79842,35.7,72.8,10.5,3.3
2,Never,Normal weight,72204,22.5,85.9,11.0,4.1
3,Former,Overweight,41770,27.4,78.8,15.4,8.0
4,Former,Obese,40936,35.6,67.5,14.2,6.6
5,Former,Normal weight,29423,22.6,80.9,13.7,8.7
6,Every day,Overweight,10357,27.3,64.4,21.4,12.6
7,Every day,Normal weight,10322,22.3,65.1,21.7,15.2
8,Every day,Obese,10062,35.9,56.9,19.3,9.9
9,Some days,Overweight,4543,27.3,75.8,26.5,10.8


In [10]:
# gold.dim_chronic_condition — chronic condition dimension grouped by total co-morbidity count
# Breaks down respondents by how many conditions they carry (0 through 5+) and shows the share
# within each tier that has each specific disease. Reveals co-morbidity clustering: respondents
# with 2+ conditions have sharply higher rates of diabetes, depression, and arthritis, reflecting
# the well-documented co-occurrence of chronic diseases. The condition_count grouping also
# serves as a disease-burden index for downstream stratified analysis in the fact table.
run_query("""
SELECT
    CAST(condition_count AS INT)                                               AS num_conditions,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CASE WHEN diabetes     = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_diabetes,
    ROUND(SUM(CASE WHEN heart_attack = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_heart_attack,
    ROUND(SUM(CASE WHEN stroke       = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_stroke,
    ROUND(SUM(CASE WHEN depression   = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_depression,
    ROUND(SUM(CASE WHEN copd         = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_copd,
    ROUND(SUM(CASE WHEN arthritis    = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_arthritis
FROM data_engineering.gold.dim_chronic_condition
GROUP BY CAST(condition_count AS INT)
ORDER BY num_conditions
""")

,num_conditions,respondents,pct_diabetes,pct_heart_attack,pct_stroke,pct_depression,pct_copd,pct_arthritis
0,0,169272,0.0,0.0,0.0,0.0,0.0,0.0
1,1,136875,10.2,2.4,1.8,22.1,2.4,35.8
2,2,81442,23.7,7.8,6.1,36.0,9.9,62.6
3,3,40787,38.7,16.5,12.2,45.6,24.8,78.6
4,4,18312,50.8,27.1,20.8,56.5,43.3,87.5
5,5,7307,63.8,40.4,32.7,65.3,61.9,91.6
6,6,2633,74.2,57.2,47.6,74.6,76.5,94.1
7,7,822,83.2,73.6,67.8,80.3,89.8,97.0
8,8,186,91.9,97.3,80.6,90.9,93.0,98.9
9,9,34,100.0,100.0,100.0,100.0,100.0,100.0


In [11]:
# gold.dim_healthcare_access — access dimension stratified by insurance type × PCP status
# Shows cost barrier rates and annual checkup rates across the full insurance type × primary
# care provider (PCP) matrix. The key finding this query surfaces: respondents without insurance
# or a regular doctor face dramatically higher cost barriers and far lower checkup rates,
# directly quantifying how structural access gaps translate into foregone preventive care —
# the mechanism through which Medicaid expansion is hypothesized to improve health outcomes.
run_query("""
SELECT
    has_insurance,
    insurance_type,
    has_pcp,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CASE WHEN cost_barrier  = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_cost_barrier,
    ROUND(SUM(CASE WHEN last_checkup = 'Within past year'
                   THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_checkup_last_year
FROM data_engineering.gold.dim_healthcare_access
WHERE has_insurance IS NOT NULL
GROUP BY has_insurance, insurance_type, has_pcp
ORDER BY respondents DESC
""")

,has_insurance,insurance_type,has_pcp,respondents,pct_cost_barrier,pct_checkup_last_year
0,Insured,1.0,Yes - Only One,84961,6.5,0.0
1,Insured,3.0,Yes - Only One,80860,3.4,0.0
2,Insured,3.0,Yes - More Than One,57960,4.2,0.0
3,Insured,1.0,Yes - More Than One,52066,7.3,0.0
4,Insured,2.0,Yes - Only One,21432,8.0,0.0
5,Insured,5.0,Yes - Only One,17858,12.8,0.0
6,Insured,1.0,No,17293,13.0,0.0
7,Not Insured,88.0,No,15671,44.2,0.0
8,Insured,2.0,Yes - More Than One,12413,9.2,0.0
9,Insured,5.0,Yes - More Than One,10266,15.1,0.0


In [12]:
# gold.dim_preventive_care — preventive care dimension reporting population-level screening uptake
# Returns a single aggregate row with nationwide uptake rates for six preventive services: flu
# and pneumococcal vaccination, mammogram (within 2 years), cervical cancer screening, colorectal
# cancer screening, and HIV testing. These national baselines serve as benchmarks for subgroup
# comparisons — e.g., testing whether states with higher SVI or no Medicaid expansion have
# significantly lower preventive care utilization rates in the fact table analysis.
run_query("""
SELECT
    COUNT(*)                                                                   AS total_respondents,
    ROUND(SUM(CASE WHEN flu_shot          = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_flu_shot,
    ROUND(SUM(CASE WHEN pneumo_vaccine    = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_pneumo_vaccine,
    ROUND(SUM(CASE WHEN mammogram_2yr     = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_mammogram,
    ROUND(SUM(CASE WHEN cervical_screen   = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_cervical_screen,
    ROUND(SUM(CASE WHEN colorectal_screen = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_colorectal_screen,
    ROUND(SUM(CASE WHEN hiv_test          = TRUE THEN 1 ELSE 0 END) * 100.0
          / COUNT(*), 1)                                                        AS pct_hiv_test
FROM data_engineering.gold.dim_preventive_care
""")

,total_respondents,pct_flu_shot,pct_pneumo_vaccine,pct_mammogram,pct_cervical_screen,pct_colorectal_screen,pct_hiv_test
0,457670,22.2,24.0,10.3,21.4,36.6,31.4


In [13]:
# gold.dim_location — geographic dimension serving as the bridge key across all three data sources
# Lists every state and territory with its FIPS code, two-letter abbreviation, and full name.
# This dimension is critical in the star schema: BRFSS respondents are keyed by state FIPS,
# SVI data is keyed by state_code (abbreviation), and Medicaid data is keyed by state_name —
# dim_location normalizes all three into a single surrogate key (location_key) that enables
# clean joins across the fact table and all policy/vulnerability dimensions.
run_query("""
SELECT
    location_key,
    state_fips,
    state_code,
    state_name
FROM data_engineering.gold.dim_location
ORDER BY state_name
""")

,location_key,state_fips,state_code,state_name
0,1,1.0,AL,Alabama
1,2,2.0,AK,Alaska
2,3,4.0,AZ,Arizona
3,4,5.0,AR,Arkansas
4,5,6.0,CA,California
5,6,8.0,CO,Colorado
6,7,9.0,CT,Connecticut
7,8,10.0,DE,Delaware
8,9,11.0,DC,District of Columbia
9,10,12.0,FL,Florida


In [14]:
# gold.dim_medicaid — Medicaid expansion policy dimension with ACA adoption timeline
# Groups states by expansion decision and year, collecting the list of states in each bucket.
# ACA-compliant expansion started in January 2014; states that adopted later or chose not to
# expand represent deliberate policy decisions with measurable downstream effects on insurance
# coverage and access that are observable when this dimension is joined to the fact table.
# COLLECT_LIST provides a human-readable roster of states in each policy category.
run_query("""
SELECT
    medicaid_expansion_status,
    expansion_year,
    COUNT(*)                 AS state_count,
    COLLECT_LIST(state_name) AS states
FROM data_engineering.gold.dim_medicaid
GROUP BY medicaid_expansion_status, expansion_year
ORDER BY medicaid_expansion_status, expansion_year
""")

,medicaid_expansion_status,expansion_year,state_count,states
0,Adopted,2014.0,27,"[""Arizona"",""Arkansas"",""California"",""Colorado"",..."
1,Adopted,2015.0,3,"[""Alaska"",""Indiana"",""Pennsylvania""]"
2,Adopted,2016.0,2,"[""Louisiana"",""Montana""]"
3,Adopted,2019.0,2,"[""Maine"",""Virginia""]"
4,Adopted,2020.0,3,"[""Idaho"",""Nebraska"",""Utah""]"
5,Adopted,2021.0,2,"[""Missouri"",""Oklahoma""]"
6,Adopted,2023.0,2,"[""North Carolina"",""South Dakota""]"
7,Not Adopted,NaN,10,"[""Alabama"",""Florida"",""Georgia"",""Kansas"",""Missi..."
8,Not Applicable,NaN,3,"[""Guam"",""Puerto Rico"",""Virgin Islands""]"


In [15]:
# gold.dim_svi — Social Vulnerability Index dimension at state level, all four CDC themes exposed
# Ranks states by overall SVI score (0 = least vulnerable, 1 = most vulnerable) and exposes
# each of the four CDC theme subscores: socioeconomic status, household composition/disability,
# minority/language access, and housing/transportation. Additional social determinant columns
# (uninsured rate, poverty rate, unemployment, education, disability, minority share) enable
# targeted analysis of which structural factors correlate most strongly with poor health outcomes
# when this dimension is joined to the fact table alongside the Medicaid expansion dimension.
run_query("""
SELECT
    state_name,
    state_code,
    svi_overall,
    svi_socioeconomic,
    svi_household,
    svi_minority,
    svi_housing_transport,
    pct_uninsured,
    pct_poverty,
    pct_unemployed,
    pct_no_hs_diploma,
    pct_disabled,
    pct_minority
FROM data_engineering.gold.dim_svi
ORDER BY svi_overall DESC
""")

,state_name,state_code,svi_overall,svi_socioeconomic,svi_household,svi_minority,svi_housing_transport,pct_uninsured,pct_poverty,pct_unemployed,pct_no_hs_diploma,pct_disabled,pct_minority
0,New Mexico,NM,0.8152,0.7510,0.7838,0.9315,0.7013,9.52,29.15,6.34,13.11,16.32,64.39
1,Nevada,NV,0.8075,0.8161,0.6208,0.8699,0.7014,11.36,21.59,7.07,12.91,12.93,53.65
2,Louisiana,LA,0.7760,0.7300,0.6975,0.7777,0.7042,8.06,28.79,6.61,13.25,15.84,42.49
3,Arizona,AZ,0.7671,0.6713,0.6398,0.8415,0.7590,10.76,21.58,5.50,11.31,13.36,47.02
4,Texas,TX,0.7667,0.7502,0.6657,0.8883,0.6482,17.57,23.17,5.26,14.95,11.70,59.88
5,Mississippi,MS,0.7528,0.7688,0.6615,0.7777,0.6129,11.83,30.46,6.52,13.74,17.17,44.12
6,California,CA,0.7232,0.6705,0.3988,0.9267,0.7960,7.09,19.99,6.47,15.74,11.00,64.81
7,Florida,FL,0.7217,0.6970,0.5404,0.7991,0.6888,12.29,21.79,5.05,10.76,13.52,48.05
8,Oklahoma,OK,0.7199,0.6718,0.7233,0.7337,0.6142,13.93,25.03,4.91,11.02,16.64,36.28
9,Arkansas,AR,0.6796,0.6197,0.7111,0.6262,0.6230,8.83,27.30,5.22,11.79,17.67,30.31


In [16]:
# gold.dim_time — survey time dimension recording interview year, month, and quarter
# Counts distinct month × quarter × year combinations present in the survey data.
# BRFSS is conducted continuously throughout the year rather than in a single wave,
# so this dimension enables time-series analysis of whether response volumes or self-reported
# health metrics shift across months or quarters — a pattern explored in depth in the
# rolling-average and LAG-based time-series query in the Advanced Analytics section below.
run_query("""
SELECT
    interview_year,
    quarter,
    interview_month,
    COUNT(*) AS time_periods
FROM data_engineering.gold.dim_time
GROUP BY interview_year, quarter, interview_month
ORDER BY interview_year, interview_month
""")

,interview_year,quarter,interview_month,time_periods
0,2024,None,NaN,1
1,2024,Q1,1.0,1
2,2024,Q1,2.0,1
3,2024,Q1,3.0,1
4,2024,Q2,4.0,1
5,2024,Q2,5.0,1
6,2024,Q2,6.0,1
7,2024,Q3,7.0,1
8,2024,Q3,8.0,1
9,2024,Q3,9.0,1


In [17]:
# gold.fact_health_response — star schema fact table joined across six dimension tables
# The central analytical query: joins fact_health_response to dim_location, dim_medicaid,
# dim_svi, dim_chronic_condition, dim_healthcare_access, and dim_behavior to compute weighted
# health outcome rates (diabetes, heart attack, depression, insurance, cost barriers, exercise)
# broken out by Medicaid expansion status × SVI vulnerability quartile. This 2×4 cross-tab
# directly tests whether states that expanded Medicaid show lower disease burden and better
# access, and whether any effect varies by underlying social vulnerability level — linking
# health policy, social determinants, and survey-measured outcomes in a single result set.
run_query("""
SELECT
    m.medicaid_expansion_status,
    CASE
        WHEN s.svi_overall < 0.25 THEN '1 - Low Vulnerability'
        WHEN s.svi_overall < 0.50 THEN '2 - Moderate Vulnerability'
        WHEN s.svi_overall < 0.75 THEN '3 - High Vulnerability'
        ELSE                           '4 - Very High Vulnerability'
    END                                                                        AS svi_quartile,
    COUNT(*)                                                                   AS respondents,
    ROUND(SUM(CAST(f.survey_weight AS DOUBLE)), 0)                            AS weighted_population,
    ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS diabetes_pct,
    ROUND(SUM(CASE WHEN c.heart_attack = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS heart_attack_pct,
    ROUND(SUM(CASE WHEN c.depression   = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS depression_pct,
    ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS insured_pct,
    ROUND(SUM(CASE WHEN a.cost_barrier  = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS cost_barrier_pct,
    ROUND(SUM(CASE WHEN b.exercise      = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
          / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                    AS exercise_pct
FROM      data_engineering.gold.fact_health_response  f
JOIN      data_engineering.gold.dim_location          l  ON f.location_key  = l.location_key
JOIN      data_engineering.gold.dim_medicaid          m  ON l.state_name    = m.state_name
JOIN      data_engineering.gold.dim_svi               s  ON l.state_code    = s.state_code
JOIN      data_engineering.gold.dim_chronic_condition c  ON f.condition_key = c.condition_key
JOIN      data_engineering.gold.dim_healthcare_access a  ON f.access_key    = a.access_key
JOIN      data_engineering.gold.dim_behavior          b  ON f.behavior_key  = b.behavior_key
WHERE     m.medicaid_expansion_status IS NOT NULL
GROUP BY  m.medicaid_expansion_status, svi_quartile
ORDER BY  m.medicaid_expansion_status, svi_quartile
""")

,medicaid_expansion_status,svi_quartile,respondents,weighted_population,diabetes_pct,heart_attack_pct,depression_pct,insured_pct,cost_barrier_pct,exercise_pct
0,Adopted,1 - Low Vulnerability,13501,1705258.0,9.71,4.49,23.88,0.0,8.31,82.06
1,Adopted,2 - Moderate Vulnerability,203207,84094500.0,11.84,4.54,23.25,0.0,10.47,78.82
2,Adopted,3 - High Vulnerability,135665,91785495.0,12.54,3.98,19.15,0.0,11.64,77.48
3,Adopted,4 - Very High Vulnerability,19190,13740491.0,12.97,4.65,21.23,0.0,14.08,76.25
4,Not Adopted,2 - Moderate Vulnerability,28588,7472905.0,11.96,4.36,23.15,0.0,10.61,77.93
5,Not Adopted,3 - High Vulnerability,34925,36029083.0,12.73,4.95,19.06,0.0,14.65,75.30
6,Not Adopted,4 - Very High Vulnerability,15277,26067043.0,13.64,4.00,21.44,0.0,17.16,74.12


---
## Advanced Analytics

The following queries demonstrate CTEs, window functions, and time-based analysis against the gold layer.

| Query | Techniques |
|---|---|
| Multi-source state health profile | CTEs, multi-dimension joins, weighted aggregations |
| State diabetes rankings | Window functions: `RANK`, `AVG OVER PARTITION BY`, `NTILE` |
| Monthly survey trends | Time-based analysis, window functions: `LAG`, cumulative `SUM`, rolling `AVG` |

In [18]:
# CTEs — multi-source state health profile combining outcomes, social context, and policy data
# Uses three named CTEs to modularize a query that would otherwise require a single unwieldy
# join chain: (1) state_health_metrics aggregates weighted disease rates and insurance coverage
# per state by joining the fact table to dim_location, dim_chronic_condition, and dim_healthcare_access;
# (2) state_social_context pulls SVI scores and social determinant percentages from dim_svi;
# (3) state_policy retrieves Medicaid expansion status and adoption year from dim_medicaid.
# The final SELECT assembles all three CTEs into a comprehensive per-state profile ordered by
# diabetes burden, making it easy to see at a glance whether high-disease states also have high
# poverty, high SVI scores, and whether they chose to expand Medicaid.
run_query("""
WITH state_health_metrics AS (
    SELECT
        l.state_name,
        l.state_code,
        COUNT(*)                                                                          AS respondents,
        ROUND(SUM(CAST(f.survey_weight AS DOUBLE)), 0)                                   AS weighted_n,
        ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS diabetes_pct,
        ROUND(SUM(CASE WHEN c.heart_attack = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS heart_attack_pct,
        ROUND(SUM(CASE WHEN c.depression   = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS depression_pct,
        ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS insured_pct,
        ROUND(SUM(CASE WHEN a.cost_barrier  = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS cost_barrier_pct
    FROM      data_engineering.gold.fact_health_response  f
    JOIN      data_engineering.gold.dim_location          l ON f.location_key  = l.location_key
    JOIN      data_engineering.gold.dim_chronic_condition c ON f.condition_key = c.condition_key
    JOIN      data_engineering.gold.dim_healthcare_access a ON f.access_key    = a.access_key
    GROUP BY  l.state_name, l.state_code
),
state_social_context AS (
    SELECT
        state_code,
        svi_overall,
        pct_uninsured,
        pct_poverty,
        pct_unemployed
    FROM data_engineering.gold.dim_svi
),
state_policy AS (
    SELECT
        state_name,
        medicaid_expansion_status,
        expansion_year
    FROM data_engineering.gold.dim_medicaid
)
SELECT
    h.state_name,
    p.medicaid_expansion_status,
    p.expansion_year,
    ROUND(s.svi_overall, 4)                  AS svi_overall,
    h.diabetes_pct,
    h.heart_attack_pct,
    h.depression_pct,
    h.insured_pct,
    h.cost_barrier_pct,
    s.pct_poverty,
    s.pct_uninsured
FROM      state_health_metrics  h
JOIN      state_social_context  s ON h.state_code = s.state_code
JOIN      state_policy          p ON h.state_name = p.state_name
ORDER BY  h.diabetes_pct DESC
""")

,state_name,medicaid_expansion_status,expansion_year,svi_overall,diabetes_pct,heart_attack_pct,depression_pct,insured_pct,cost_barrier_pct,pct_poverty,pct_uninsured
0,West Virginia,Adopted,2014.0,0.4505,18.34,6.98,30.07,0.0,13.02,26.92,6.39
1,Kentucky,Adopted,2014.0,0.5199,16.14,6.54,28.19,0.0,12.22,25.64,5.86
2,Louisiana,Adopted,2016.0,0.7760,15.38,5.08,26.39,0.0,14.74,28.79,8.06
3,Arkansas,Adopted,2014.0,0.6796,15.29,5.86,24.98,0.0,15.17,27.30,8.83
4,Mississippi,Not Adopted,NaN,0.7528,15.18,4.55,19.53,0.0,14.67,30.46,11.83
5,Alabama,Not Adopted,NaN,0.6243,15.04,5.51,24.69,0.0,14.03,25.36,9.55
6,Indiana,Adopted,2015.0,0.4567,14.20,5.25,24.36,0.0,11.45,20.70,7.79
7,South Carolina,Not Adopted,NaN,0.6526,13.87,4.99,20.60,0.0,13.75,23.41,10.16
8,Michigan,Adopted,2014.0,0.4470,13.57,5.40,26.57,0.0,11.23,21.05,5.19
9,Nevada,Adopted,2014.0,0.8075,13.52,4.37,20.47,0.0,16.66,21.59,11.36


In [19]:
# Window functions — state diabetes rankings within and across Medicaid expansion groups
# Demonstrates four window function patterns over a CTE-derived state outcomes table:
# (1) RANK() OVER() for a national ranking by weighted diabetes rate (rank 1 = highest burden);
# (2) RANK() OVER(PARTITION BY medicaid_expansion_status) for within-group peer ranking so you
# can see which states are outliers relative to their policy peers; (3) AVG() OVER(PARTITION BY)
# for each group's mean as a comparison baseline; and (4) NTILE(4) to assign states to national
# diabetes quartiles. The deviation_from_group_avg column isolates state-specific factors from
# the group-level Medicaid expansion effect, helping distinguish policy impact from confounding.
run_query("""
WITH state_outcomes AS (
    SELECT
        l.state_name,
        m.medicaid_expansion_status,
        ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS diabetes_pct,
        ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS insured_pct,
        ROUND(SUM(CASE WHEN a.cost_barrier  = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS cost_barrier_pct
    FROM      data_engineering.gold.fact_health_response  f
    JOIN      data_engineering.gold.dim_location          l ON f.location_key  = l.location_key
    JOIN      data_engineering.gold.dim_medicaid          m ON l.state_name    = m.state_name
    JOIN      data_engineering.gold.dim_chronic_condition c ON f.condition_key = c.condition_key
    JOIN      data_engineering.gold.dim_healthcare_access a ON f.access_key    = a.access_key
    WHERE     m.medicaid_expansion_status IS NOT NULL
    GROUP BY  l.state_name, m.medicaid_expansion_status
)
SELECT
    state_name,
    medicaid_expansion_status,
    diabetes_pct,
    insured_pct,
    cost_barrier_pct,
    RANK()  OVER(ORDER BY diabetes_pct DESC)                                              AS national_diabetes_rank,
    RANK()  OVER(PARTITION BY medicaid_expansion_status ORDER BY diabetes_pct DESC)       AS rank_within_medicaid_group,
    ROUND(AVG(diabetes_pct) OVER(PARTITION BY medicaid_expansion_status), 2)              AS group_avg_diabetes_pct,
    ROUND(diabetes_pct - AVG(diabetes_pct) OVER(PARTITION BY medicaid_expansion_status), 2) AS deviation_from_group_avg,
    NTILE(4) OVER(ORDER BY diabetes_pct DESC)                                             AS national_diabetes_quartile
FROM  state_outcomes
ORDER BY national_diabetes_rank
""")

,state_name,medicaid_expansion_status,diabetes_pct,insured_pct,cost_barrier_pct,national_diabetes_rank,rank_within_medicaid_group,group_avg_diabetes_pct,deviation_from_group_avg,national_diabetes_quartile
0,West Virginia,Adopted,18.34,0.0,13.02,1,1,11.82,6.52,1
1,Puerto Rico,Not Applicable,17.77,0.0,9.77,2,1,16.27,1.50,1
2,Virgin Islands,Not Applicable,17.72,0.0,14.48,3,2,16.27,1.45,1
3,Kentucky,Adopted,16.14,0.0,12.22,4,2,11.82,4.32,1
4,Louisiana,Adopted,15.38,0.0,14.74,5,3,11.82,3.56,1
5,Arkansas,Adopted,15.29,0.0,15.17,6,4,11.82,3.47,1
6,Mississippi,Not Adopted,15.18,0.0,14.67,7,1,12.97,2.21,1
7,Alabama,Not Adopted,15.04,0.0,14.03,8,2,12.97,2.07,1
8,Indiana,Adopted,14.20,0.0,11.45,9,5,11.82,2.38,1
9,South Carolina,Not Adopted,13.87,0.0,13.75,10,3,12.97,0.90,1


In [20]:
# Time-based analysis — monthly survey response trends with rolling averages and LAG comparisons
# Joins fact_health_response to dim_time, dim_chronic_condition, and dim_healthcare_access to
# compute monthly response volumes and weighted health rates across the 2024 survey year.
# Four window functions operate over the monthly CTE: (1) cumulative SUM() OVER() for a running
# response total; (2) LAG() to retrieve the prior month's count; (3) month-over-month percent
# change derived from the LAG (NULLIF guards against divide-by-zero in the first month); and
# (4) a 3-month rolling AVG() diabetes rate that smooths sampling noise to reveal true seasonal
# trends. Together these show whether BRFSS participation or self-reported disease rates shift
# meaningfully across calendar months.
run_query("""
WITH monthly_stats AS (
    SELECT
        t.interview_year,
        t.interview_month,
        t.quarter,
        COUNT(*)                                                                          AS monthly_responses,
        ROUND(SUM(CAST(f.survey_weight AS DOUBLE)), 0)                                   AS weighted_respondents,
        ROUND(SUM(CASE WHEN c.diabetes     = TRUE THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS monthly_diabetes_pct,
        ROUND(SUM(CASE WHEN a.has_insurance = 'Yes' THEN CAST(f.survey_weight AS DOUBLE) ELSE 0 END)
              / SUM(CAST(f.survey_weight AS DOUBLE)) * 100, 2)                           AS monthly_insured_pct
    FROM      data_engineering.gold.fact_health_response  f
    JOIN      data_engineering.gold.dim_time              t ON f.time_key      = t.time_key
    JOIN      data_engineering.gold.dim_chronic_condition c ON f.condition_key = c.condition_key
    JOIN      data_engineering.gold.dim_healthcare_access a ON f.access_key    = a.access_key
    GROUP BY  t.interview_year, t.interview_month, t.quarter
)
SELECT
    interview_year,
    interview_month,
    quarter,
    monthly_responses,
    weighted_respondents,
    monthly_diabetes_pct,
    monthly_insured_pct,
    SUM(monthly_responses)  OVER(ORDER BY interview_year, interview_month
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)      AS cumulative_responses,
    LAG(monthly_responses)  OVER(ORDER BY interview_year, interview_month)               AS prev_month_responses,
    ROUND(
        (monthly_responses - LAG(monthly_responses) OVER(ORDER BY interview_year, interview_month))
        * 100.0
        / NULLIF(LAG(monthly_responses) OVER(ORDER BY interview_year, interview_month), 0),
    1)                                                                                    AS mom_response_pct_change,
    ROUND(AVG(monthly_diabetes_pct) OVER(ORDER BY interview_year, interview_month
                                          ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2)  AS rolling_3mo_diabetes_pct
FROM  monthly_stats
ORDER BY interview_year, interview_month
""")

,interview_year,interview_month,quarter,monthly_responses,weighted_respondents,monthly_diabetes_pct,monthly_insured_pct,cumulative_responses,prev_month_responses,mom_response_pct_change,rolling_3mo_diabetes_pct
0,2024,1,Q1,9224,5157389.0,12.43,0.0,9224,NaN,None,12.43
1,2024,2,Q1,34086,18642806.0,11.98,0.0,43310,9224.0,269.5,12.21
2,2024,3,Q1,37958,22772512.0,12.54,0.0,81268,34086.0,11.4,12.32
3,2024,4,Q2,37039,22507620.0,12.43,0.0,118307,37958.0,-2.4,12.32
4,2024,5,Q2,40966,23616653.0,13.12,0.0,159273,37039.0,10.6,12.70
5,2024,6,Q2,43623,22817476.0,12.25,0.0,202896,40966.0,6.5,12.60
6,2024,7,Q3,44360,28866486.0,12.77,0.0,247256,43623.0,1.7,12.71
7,2024,8,Q3,42175,27181317.0,12.39,0.0,289431,44360.0,-4.9,12.47
8,2024,9,Q3,34750,19582324.0,12.73,0.0,324181,42175.0,-17.6,12.63
9,2024,10,Q4,38167,23100645.0,12.16,0.0,362348,34750.0,9.8,12.43
